In [10]:
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from tqdm import tqdm

In [2]:
loaded = torch.load("../dummy_frame.pt")

images = [loaded['image']]
targets = [loaded['targets']]

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(1, 1)):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_ch,
                out_ch,
                kernel_size=3,
                stride=stride,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SalsaXYRegressor(nn.Module):
    """
    Input:
        (B, 4, T=8, F=128)

    Output:
        (B, 2)

        pred[:,0] = x coordinate [0,359]
        pred[:,1] = y coordinate [0,179]
    """

    def __init__(self):
        super().__init__()

        self.enc0 = ConvBlock(4, 64)

        self.enc1 = ConvBlock(
            64,
            128,
            stride=(1, 2),
        )

        self.enc2 = ConvBlock(
            128,
            256,
            stride=(1, 2),
        )

        self.enc3 = ConvBlock(
            256,
            512,
            stride=(1, 2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.regressor = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),

            nn.Linear(256, 128),
            nn.ReLU(inplace=True),

            nn.Linear(128, 2),
        )

    def forward(self, x):

        x = self.enc0(x)
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)

        x = self.pool(x)
        x = x.flatten(1)

        xy = self.regressor(x)

        return xy

In [4]:
vmap = targets[0]["vmap"]

y, x = torch.where(
    vmap == vmap.max()
)

gt_xy = torch.tensor(
    [x[0], y[0]],
    dtype=torch.float32
)

In [5]:
gt_xy

tensor([97., 89.])

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SalsaXYRegressor().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

# --------------------------------------------------
# Fetch ONE sample
# --------------------------------------------------

image, target = images[0], targets[0]

image = image.unsqueeze(0).to(device)   # (1,4,8,128)

vmap = target["vmap"]

# peak coordinate
y, x = torch.where(vmap == vmap.max())

gt_xy = torch.tensor(
    [[float(x[0]), float(y[0])]],
    dtype=torch.float32,
    device=device,
)

print("GT XY:", gt_xy)

# --------------------------------------------------
# Training
# --------------------------------------------------

epochs = 100

pbar = tqdm(range(epochs))

for epoch in pbar:

    model.train()

    pred_xy = model(image)

    loss = F.smooth_l1_loss(
        pred_xy,
        gt_xy,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:

        px = pred_xy[0, 0].item()
        py = pred_xy[0, 1].item()

        gx = gt_xy[0, 0].item()
        gy = gt_xy[0, 1].item()

        pbar.set_description(
            f"loss={loss.item():.4f} "
            f"pred=({px:.1f},{py:.1f}) "
            f"gt=({gx:.1f},{gy:.1f})"
        )

print("\nFinished")

with torch.no_grad():

    pred_xy = model(image)

    print("\nFinal Prediction:")
    print("Pred:", pred_xy.cpu().numpy())
    print("GT  :", gt_xy.cpu().numpy())

GT XY: tensor([[97., 89.]])


loss=92.5211 pred=(0.0,-0.1) gt=(97.0,89.0):   6%|▌         | 6/100 [00:00<00:05, 17.05it/s]

loss=0.0354 pred=(97.3,89.2) gt=(97.0,89.0): 100%|██████████| 100/100 [00:05<00:00, 17.79it/s]


Finished

Final Prediction:
Pred: [[97.10749 89.06426]]
GT  : [[97. 89.]]
